# LLM Retraining for Customer Support (Simulated Case)

**Author:** Karthik Prakhya, Spandana Prakhya  
**Company:** OptiML Data Analysis AB  
**Website:** https://optimldataanalysis.se  
**Contact:** info@optimldataanalysis.se

**License:** MIT License (see LICENSE file)  
**Disclaimer:** This notebook is for educational and demonstration purposes only.  
It uses synthetic, non-confidential data to illustrate how we would fine‑tune and evaluate
a language model for customer‑support scenarios. In real client projects, we replace
the toy data and small model with your actual policies, FAQs and a stronger base model.


## 1. Executive Summary

This notebook demonstrates, end‑to‑end, how we can **adapt a base LLM to a client's
customer‑support domain** using lightweight fine‑tuning (LoRA) and then **evaluate**
it on realistic support questions.

What we show here:

- How to represent **support policies and FAQs** as training data
- How to apply **parameter‑efficient fine‑tuning (LoRA)** on top of an open‑source model
- How to run a small **evaluation suite** of realistic customer questions
- How to interpret both **automatic metrics** and **qualitative examples**

What is intentionally simplified for the demo:

- We use a **small open‑source model** (`facebook/opt-125m`) instead of a large production model
- We train on a **small synthetic dataset** that mimics typical support flows
- We run everything in a single notebook; in production this would be automated via CI/CD and cloud training

For clients, this notebook acts as a **conceptual blueprint**:

> In a real engagement, we plug in your data, policies and infrastructure,
> but the workflow (data → fine‑tune → evaluate → deploy) remains the same.


# 2. Case Study: Retraining LLMs for Customer Support Automation

In a typical project, we start from a **general‑purpose language model** and adapt it
to a client's customer‑support domain:

- CRM / ticket data
- Knowledge‑base articles
- Internal policies (returns, billing, SLAs, data privacy)
- Tone‑of‑voice and compliance guidelines

**Cloud integration (example stack):**

- Training & hosting on AWS SageMaker
- Artifacts stored in S3
- CI/CD with GitHub Actions for regular re‑training and safe roll‑outs

This notebook focuses on the core modeling steps:

1. Build a **synthetic support dataset** (for demo purposes)
2. Apply **LoRA** to fine‑tune the model efficiently
3. Evaluate on **realistic support scenarios** with simple metrics and qualitative review
4. Sketch how this plugs into a **production deployment** (SageMaker endpoint)


## 3. Mathematical background

We fine-tune a pretrained causal language model (LLM) on a supervised dataset of (prompt, response) pairs.  
The primary training objective is the **causal language modeling cross-entropy** loss:

$$ \mathcal{L}_{CE}(\theta) = -\sum_{t=1}^{T} \log p_\theta(x_t\mid x_{<t}) $$

where $x_{<t}$ are previous tokens and $\theta$ are model parameters.  
Training updates use gradients of this loss and an optimizer such as AdamW.

To avoid catastrophic forgetting and encourage the fine-tuned model to remain close to the pretrained policy, people often add a **KL-divergence** or **L2 regularization** term to the objective:

$$ \mathcal{L}(\theta) = \mathcal{L}_{CE}(\theta) + \lambda_{KL} D_{KL}(p_{\theta_0} \| p_{\theta}) $$

### Parameter-efficient fine-tuning (LoRA)

LoRA (Low-Rank Adaptation) injects low-rank matrices into each weight update of selected layers (usually attention projection matrices).  
For a weight matrix $W_0 \in \mathbb{R}^{d\times k}$ we parametrize the update as:

$$ W = W_0 + \Delta W, \quad \Delta W = BA $$

with $B \in \mathbb{R}^{d\times r},\ A \in \mathbb{R}^{r\times k}$ and $r \ll \min(d,k)$.  
During fine-tuning we only learn $A,B$ (and optionally a small subset of original parameters), drastically reducing trainable parameters and GPU memory.

### Evaluation metrics

- Perplexity (PPL): $$\mathrm{PPL} = \exp\left(\frac{1}{N}\sum_{i} \mathcal{L}_{CE}(x^{(i)})\right)$$
- BLEU/ROUGE or task-specific accuracy for structured responses
- Human evaluation (satisfaction, correctness) for production systems


In [9]:
# Environment setup (install required libraries before running)
# !pip install -U transformers datasets peft accelerate safetensors sagemaker boto3

print("Install the packages above before attempting training on GPU/SageMaker.")


Install the packages above before attempting training on GPU/SageMaker.


In [10]:
# Reproducibility: fix random seeds (best‑effort)
import random
import numpy as np
import torch

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("Random seeds set to", SEED)


Random seeds set to 42


In [11]:
from datasets import Dataset

# 4. Synthetic customer‑support dataset for this demo
# --------------------------------------------------
# In real projects we would ingest client FAQs, policies and historical tickets.
# Here we create a small but realistic synthetic dataset of (prompt, response) pairs.

examples = [
    # Authentication / account access
    {
        "prompt": "Customer: I forgot my password and cannot log in. What should I do?\nAgent:",
        "response": "You can reset your password by clicking \"Forgot password\" on the login page. We will send a reset link to your registered email address. If you don't see it within a few minutes, please check your spam folder or contact our support team again."
    },
    {
        "prompt": "Customer: I changed my email address and now I can't log in. How do I fix this?\nAgent:",
        "response": "If you recently changed your email address, try logging in with your previous email. Once logged in, you can update your email under Account settings -> Profile. If you no longer have access to the old email, contact our support team with a recent invoice so we can verify your identity."
    },
    {
        "prompt": "Customer: My account is locked after too many failed login attempts. How can I unlock it?\nAgent:",
        "response": "For security reasons, your account is temporarily locked after several failed login attempts. Please wait 15 minutes and try again, or use the \"Forgot password\" link on the login page to reset your password immediately."
    },

    # Opening hours / contact
    {
        "prompt": "Customer: What are your customer support opening hours on weekdays?\nAgent:",
        "response": "Our customer support team is available Monday to Friday from 09:00 to 17:00 CET. If you contact us outside these hours, we will follow up on the next business day."
    },
    {
        "prompt": "Customer: Do you offer support on weekends or public holidays?\nAgent:",
        "response": "Our standard support is available Monday to Friday 09:00–17:00 CET. On weekends and public holidays we only handle urgent incidents for business customers on the Premium plan."
    },

    # Returns & refunds
    {
        "prompt": "Customer: I received the wrong item. How can I return it?\nAgent:",
        "response": "I'm sorry about that. You can create a free return label from your account under \"Orders\" -> \"Return item\". Please pack the item securely and drop it off at the indicated parcel shop within 30 days. Once we receive it, we will ship the correct item or issue a refund, depending on your choice."
    },
    {
        "prompt": "Customer: The product is damaged. Can I get a replacement?\nAgent:",
        "response": "Yes, of course. Please upload one or two photos of the damage under your order details and choose \"Item arrived damaged\" as the reason. We will review the case and send a replacement or refund according to your preference."
    },
    {
        "prompt": "Customer: How long does it take to receive my refund after I return an item?\nAgent:",
        "response": "Once we receive your returned item in our warehouse, refunds are usually processed within 5–7 business days. The amount will be credited back to the same payment method you used for the purchase."
    },

    # Shipping & tracking
    {
        "prompt": "Customer: My order hasn't arrived yet. How can I see where it is?\nAgent:",
        "response": "You can track your shipment from your account under \"Orders\" by clicking \"Track package\". There you will see the latest scan from the carrier. If the status hasn't changed for more than 48 hours, please contact us and we will investigate with the carrier."
    },
    {
        "prompt": "Customer: The tracking link says delivered but I didn't receive my package. What should I do?\nAgent:",
        "response": "Please first check if a neighbour or reception has received the package on your behalf. If you still can't find it, contact us within 7 days via your order page. We will open an investigation with the carrier and help you with a replacement or refund."
    },

    # Order changes / cancellation
    {
        "prompt": "Customer: I just placed an order by mistake. Can I cancel it?\nAgent:",
        "response": "If the order has not been shipped yet, you can cancel it directly from your account under \"Orders\" -> \"Cancel order\". If shipping has already started, we may not be able to stop it, but you can use our free return process once it arrives."
    },
    {
        "prompt": "Customer: Can I change the delivery address after placing an order?\nAgent:",
        "response": "You can change the delivery address as long as the order has not yet been handed over to the carrier. Go to \"Orders\", open the relevant order and click \"Edit delivery address\". If that option is not available, please contact the carrier directly once the parcel is in transit."
    },

    # Subscription & billing
    {
        "prompt": "Customer: I was charged for my subscription but I thought it was cancelled.\nAgent:",
        "response": "I'm sorry for the confusion. Please go to \"Account settings\" -> \"Subscriptions\" to check the status of your plan. If it is still active, you can cancel it with immediate effect there. If you believe you were charged by mistake, share the invoice number with us and we will review a refund."
    },
    {
        "prompt": "Customer: How can I download invoices for my subscription payments?\nAgent:",
        "response": "You can download all invoices from your account under \"Billing\" -> \"Invoices\". Each invoice can be downloaded as a PDF for your records or accounting."
    },
    {
        "prompt": "Customer: Can I change my payment method for the subscription?\nAgent:",
        "response": "Yes. Go to \"Account settings\" -> \"Payment methods\" to add or update your card or bank details. Changes will apply to future subscription renewals."
    },

    # Data privacy / account management
    {
        "prompt": "Customer: How do I delete my account and all my personal data?\nAgent:",
        "response": "You can request deletion of your account and personal data in \"Account settings\" -> \"Privacy\" -> \"Delete account\". Once you confirm, we will permanently delete your personal data within 30 days, except for information we are legally required to keep (for example, invoices)."
    },
    {
        "prompt": "Customer: Where can I see what personal data you store about me?\nAgent:",
        "response": "You can download a copy of your personal data under \"Account settings\" -> \"Privacy\" -> \"Download my data\". The export includes information such as your profile details, orders and support interactions."
    },

    # Technical issues
    {
        "prompt": "Customer: Your website keeps showing an error when I try to pay. What can I do?\nAgent:",
        "response": "I'm sorry for the trouble. First, please try clearing your browser cache or using a different browser or device. If the problem persists, send us a screenshot of the error message and the time it occurred so we can investigate it with our technical team."
    },
    {
        "prompt": "Customer: The mobile app crashes when I open my order history. How do I fix this?\nAgent:",
        "response": "Please make sure you are using the latest version of the app from the App Store or Google Play. If the issue continues, try reinstalling the app. If it still crashes, contact us with your device model and OS version so we can forward the details to our developers."
    },

    # B2B / SLA / discounts
    {
        "prompt": "Customer: What is your response time for business customers on the premium plan?\nAgent:",
        "response": "For business customers on the Premium plan, our service-level agreement guarantees an initial response within 2 business hours during support opening times. Complex cases may take longer to resolve, but we will keep you informed about the progress."
    },
    {
        "prompt": "Customer: Do you offer volume discounts if we move all our teams to your platform?\nAgent:",
        "response": "Yes, we offer tiered volume discounts for teams. The exact discount depends on the number of seats and contract length. If you share an estimate of how many users you plan to onboard, our sales team can prepare a tailored offer for you."
    },
    {
        "prompt": "Customer: Can you sign a data processing agreement (DPA) for our company?\nAgent:",
        "response": "Yes, we can sign a data processing agreement for business customers. Please contact our sales or legal team and they will share our standard DPA template or review your company-specific requirements."
    },
]

ds = Dataset.from_list([
    {"text": ex["prompt"] + " " + ex["response"]}
    for ex in examples
])
print("Example training row:", ds[0])
print("Total training examples:", len(ds))


Example training row: {'text': 'Customer: I forgot my password and cannot log in. What should I do?\nAgent: You can reset your password by clicking "Forgot password" on the login page. We will send a reset link to your registered email address. If you don\'t see it within a few minutes, please check your spam folder or contact our support team again.'}
Total training examples: 22


In [12]:
from transformers import AutoTokenizer

# 5. Tokenization
# ---------------
# We tokenize the synthetic texts for causal language modeling.

# NOTE: choose a local-small model or a model hub id when running for real
model_name = "facebook/opt-125m"  # placeholder for demo

tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize(example):
    return tokenizer(example["text"], truncation=True, max_length=512)

ds_tok = ds.map(tokenize, batched=False)
print("Tokenization complete. Example keys:", ds_tok[0].keys())


Map: 100%|█████████████████████████████| 22/22 [00:00<00:00, 1459.90 examples/s]

Tokenization complete. Example keys: dict_keys(['text', 'input_ids', 'attention_mask'])


In [15]:
# --------------------------------------------------------------
# 6. SAFE & ROBUST LoRA TRAINING BLOCK FOR CAUSAL LANGUAGE MODELS
# --------------------------------------------------------------
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    Trainer,
    TrainingArguments,
    DataCollatorForLanguageModeling
)
from peft import LoraConfig, get_peft_model
import torch
from datasets import Dataset

# -----------------------------
# Load base model + tokenizer
# -----------------------------
model_name = "facebook/opt-125m"

tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token  # required for OPT

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto"
)

# -----------------------------
# Apply LoRA
# -----------------------------
lora = LoraConfig(
    r=8,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora)


# ------------------------------------------------
# 1) DATA CLEANING — PREVENTS ALL COLLATION ERRORS
# ------------------------------------------------
def clean_text(x):
    """Remove empty, weird, or malformed samples."""
    if ("text" not in x) or (x["text"] is None):
        return False
    if len(x["text"].strip()) == 0:
        return False
    if not isinstance(x["text"], str):
        return False
    return True

ds_clean = ds.filter(clean_text)


# ------------------------------------------------
# 2) TOKENIZATION — GUARANTEED SAFE
# ------------------------------------------------
def tokenize(example):
    return tokenizer(
        example["text"],
        truncation=True,
        max_length=512
    )

ds_tok = ds_clean.map(tokenize, batched=False)


# ------------------------------------------------
# 3) (Optional) Additional dataset sanity checks could go here
# ------------------------------------------------

# ------------------------------------------------
# 4) COLLATOR — THE *CORRECT* ONE FOR CAUSAL LM — THE *CORRECT* ONE FOR CAUSAL LM
# ------------------------------------------------
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False,
    pad_to_multiple_of=8          # ensures even padding ‣ prevents dimErrors
)


# ------------------------------------------------
# 5) TRAINING CONFIG — RELIABLE SETTINGS
# ------------------------------------------------
# training_args = TrainingArguments(
#    output_dir="./outputs/llm-lora-demo",
#    per_device_train_batch_size=4,
#    gradient_accumulation_steps=8,
#    num_train_epochs=3,
#    learning_rate=2e-4,
#    fp16=True,
#    logging_steps=10,
#    save_total_limit=3,
# )
training_args = TrainingArguments(
    output_dir="./outputs/llm-lora-demo",
    overwrite_output_dir=True,
    num_train_epochs=10,               # more passes over the small dataset
    per_device_train_batch_size=2,     # small batch is fine
    gradient_accumulation_steps=2,     # effective batch size 4
    learning_rate=2e-4,
    warmup_ratio=0.05,
    weight_decay=0.01,
    logging_steps=5,
    save_total_limit=1,
    fp16=True,
    report_to=[],                      # disable WandB etc. in simple notebooks
)


# ------------------------------------------------
# 6) TRAINER — BULLETPROOF
# ------------------------------------------------
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=ds_tok,
    tokenizer=tokenizer,
    data_collator=data_collator
)


# ------------------------------------------------
# 🚀 TRAIN — Guaranteed not to crash now
# ------------------------------------------------
trainer.train()

print("FINISHED: Training completed successfully without dimension errors.")



Map: 100%|█████████████████████████████| 22/22 [00:00<00:00, 1093.15 examples/s]
/tmp/ipykernel_13866/77488576.py:119: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
The model is already on multiple devices. Skipping the move to device specified in `args`.
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 2}.


Step,Training Loss
5,3.070600
10,2.885300
15,2.620000
20,2.598900
25,2.421000
30,2.384800
35,2.338000
40,2.313400
45,2.364800
50,2.215200


FINISHED: Training completed successfully without dimension errors.


## 7. Evaluation: metrics and qualitative examples

In production, we combine **automatic metrics** with **human review**:

- Automatic metrics (here: a simple token‑overlap F1) give a quick, quantitative signal.
- Qualitative examples let us see if the model follows the **correct flows and tone**.

Below we:

1. Define a small evaluation suite of realistic customer questions.
2. Generate answers with the fine‑tuned model using safe decoding settings.
3. Compute a rough F1 score per scenario and on average.
4. Show a few **hero examples** side‑by‑side (Customer / Model / Reference) for discussion.


In [16]:
# 7.1 Evaluation setup: pipeline, scenarios, F1 metric
from transformers import pipeline
import re
from textwrap import shorten
from typing import List, Dict

device = 0 if torch.cuda.is_available() else -1
print(f"Device set to use {'cuda:0' if device == 0 else 'CPU'}")

gen_pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer
)

# Evaluation scenarios (mirrors the intents we care about)
eval_scenarios: List[Dict[str, str]] = [
    {
        "intent": "reset_password",
        "prompt": "Customer: I can't log in because I forgot my password. What should I do?\nAgent:",
        "reference": "You can reset your password by clicking \"Forgot password\" on the login page. We will send a reset link to your registered email address. If you don't see it within a few minutes, please check your spam folder or contact our support team again."
    },
    {
        "intent": "opening_hours",
        "prompt": "Customer: What are your customer support opening hours on weekdays?\nAgent:",
        "reference": "Our customer support team is available Monday to Friday from 09:00 to 17:00 CET. If you contact us outside these hours, we will follow up on the next business day."
    },
    {
        "intent": "returns_policy",
        "prompt": "Customer: I received the wrong item. How can I return it?\nAgent:",
        "reference": "I'm sorry about that. You can create a free return label from your account under \"Orders\" -> \"Return item\". Please pack the item securely and drop it off at the indicated parcel shop within 30 days. Once we receive it, we will ship the correct item or issue a refund, depending on your choice."
    },
    {
        "intent": "shipping_status",
        "prompt": "Customer: My order hasn't arrived yet. How can I see where it is?\nAgent:",
        "reference": "You can track your shipment from your account under \"Orders\" by clicking \"Track package\". There you will see the latest scan from the carrier. If the status hasn't changed for more than 48 hours, please contact us and we will investigate with the carrier."
    },
    {
        "intent": "cancel_order",
        "prompt": "Customer: I just placed an order by mistake. Can I cancel it?\nAgent:",
        "reference": "If the order has not been shipped yet, you can cancel it directly from your account under \"Orders\" -> \"Cancel order\". If shipping has already started, we may not be able to stop it, but you can use our free return process once it arrives."
    },
    {
        "intent": "subscription_billing",
        "prompt": "Customer: I was charged for my subscription but I thought it was cancelled.\nAgent:",
        "reference": "I'm sorry for the confusion. Please go to \"Account settings\" -> \"Subscriptions\" to check the status of your plan. If it is still active, you can cancel it with immediate effect there. If you believe you were charged by mistake, share the invoice number with us and we will review a refund."
    },
    {
        "intent": "data_privacy",
        "prompt": "Customer: How do I delete my account and all my personal data?\nAgent:",
        "reference": "You can request deletion of your account and personal data in \"Account settings\" -> \"Privacy\" -> \"Delete account\". Once you confirm, we will permanently delete your personal data within 30 days, except for information we are legally required to keep (for example, invoices)."
    },
    {
        "intent": "technical_issue",
        "prompt": "Customer: Your website keeps showing an error when I try to pay. What can I do?\nAgent:",
        "reference": "I'm sorry for the trouble. First, please try clearing your browser cache or using a different browser or device. If the problem persists, send us a screenshot of the error message and the time it occurred so we can investigate it with our technical team."
    },
    {
        "intent": "sla_question",
        "prompt": "Customer: What is your response time for business customers on the premium plan?\nAgent:",
        "reference": "For business customers on the Premium plan, our service-level agreement guarantees an initial response within 2 business hours during support opening times. Complex cases may take longer to resolve, but we will keep you informed about the progress."
    },
    {
        "intent": "discount_policy",
        "prompt": "Customer: Do you offer volume discounts if we move all our teams to your platform?\nAgent:",
        "reference": "Yes, we offer tiered volume discounts for teams. The exact discount depends on the number of seats and contract length. If you share an estimate of how many users you plan to onboard, our sales team can prepare a tailored offer for you."
    },
]

def generate_answer(prompt: str) -> str:
    """Generate a concise answer using sampling with repetition controls."""
    out = gen_pipe(
        prompt,
        max_new_tokens=80,
        do_sample=True,
        top_p=0.9,
        temperature=0.7,
        repetition_penalty=1.2,
        no_repeat_ngram_size=4,
        eos_token_id=tokenizer.eos_token_id,
        pad_token_id=tokenizer.eos_token_id,
    )[0]["generated_text"]

    if "Agent:" in out:
        answer = out.split("Agent:", 1)[-1].strip()
    else:
        answer = out.strip()
    # Show only the first paragraph for readability
    return answer.split("\n\n")[0].strip()

def normalize(text: str):
    return re.findall(r"\w+", text.lower())

def token_f1(pred: str, ref: str) -> float:
    pred_tokens = normalize(pred)
    ref_tokens = normalize(ref)
    if not pred_tokens or not ref_tokens:
        return 0.0
    pred_set, ref_set = set(pred_tokens), set(ref_tokens)
    overlap = len(pred_set & ref_set)
    precision = overlap / len(pred_set)
    recall = overlap / len(ref_set)
    if precision == 0.0 or recall == 0.0:
        return 0.0
    return 2 * precision * recall / (precision + recall)

print("Running evaluation on", len(eval_scenarios), "scenarios...\n")

results = []
for item in eval_scenarios:
    pred = generate_answer(item["prompt"])
    ref = item["reference"]
    f1_score = token_f1(pred, ref)
    results.append({"intent": item["intent"], "f1": f1_score, "pred": pred, "ref": ref, "prompt": item["prompt"]})
    print("INTENT :", item["intent"])
    print("PRED   :", shorten(pred, width=260, placeholder=" ..."))
    print("REF    :", shorten(ref, width=260, placeholder=" ..."))
    print(f"F1     : {f1_score:.3f}")
    print("-" * 80)

if results:
    avg_f1 = sum(r["f1"] for r in results) / len(results)
    print(f"\nAverage token-overlap F1 across {len(results)} scenarios: {avg_f1:.3f}")
    print("Note: this is a very rough automatic metric; human review is still essential.")


Device set to use cuda:0


Device set to use cuda:0
Running evaluation on 10 scenarios...

INTENT : reset_password
PRED   : Please login and enter your username for the next day to allow us to reset it. Your email address will be sent a confirmation message about how you can reset your password. Once this has happened, we'll send you an update with our new password.
REF    : You can reset your password by clicking "Forgot password" on the login page. We will send a reset link to your registered email address. If you don't see it within a few minutes, please check your spam folder or contact our support team again.
F1     : 0.430
--------------------------------------------------------------------------------
INTENT : opening_hours
PRED   : "I'm not sure how to open a new account" or "How to open a current account from within the app", please contact me via email. I can give you details of when we will be opening our day and timezone. We will provide an example of how long we plan to operate ...
REF    : Our cust

In [17]:
# 7.2 Hero examples: side-by-side for discussion

hero_intents = ["reset_password", "subscription_billing", "data_privacy", "discount_policy"]

print("\nSample qualitative evaluation (hero scenarios):\n")
for r in results:
    if r["intent"] not in hero_intents:
        continue
    customer = r["prompt"].split("Customer:", 1)[-1].split("Agent:")[0].strip()
    print(f"Intent   : {r['intent']}")
    print("Customer :", customer)
    print("Model    :", r["pred"])
    print("Reference:", r["ref"])
    print("-" * 80)



Sample qualitative evaluation (hero scenarios):

Intent   : reset_password
Customer : I can't log in because I forgot my password. What should I do?
Model    : Please login and enter your username for the next day to allow us to reset it. Your email address will be sent a confirmation message about how you can reset your password. Once this has happened, we'll send you an update with our new password.
Reference: You can reset your password by clicking "Forgot password" on the login page. We will send a reset link to your registered email address. If you don't see it within a few minutes, please check your spam folder or contact our support team again.
--------------------------------------------------------------------------------
Intent   : subscription_billing
Customer : I was charged for my subscription but I thought it was cancelled.
Model    : Can you confirm how your account has been suspended or canceled?
 Agent: Yes, your account will be suspended if you refuse to cancel the s

## Retrieval-Augmented Generation (RAG) demo

So far, the model answers purely from what it learned during fine-tuning.
In a production system, we usually **combine the model with a knowledge base**:

- Policies / FAQs are stored as text documents.
- At inference time we **retrieve** the most relevant documents.
- We ask the model to answer using **only that retrieved context**.

This pattern is called **Retrieval-Augmented Generation (RAG)**.

Below is a lightweight, self-contained RAG demo using TF‑IDF + cosine similarity
as the retriever (no external services required). In a real project we would swap
this for a vector database and more advanced retrieval, but the overall logic
remains the same.


In [18]:
# Simple RAG implementation using TF-IDF + cosine similarity
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
from transformers import pipeline

# 1) Build a tiny knowledge base (KB)
# Here we reuse the synthetic responses as "policy snippets".
# In a real system, you would load help-center articles, policy docs, etc.
kb_texts = []
kb_ids = []

try:
    # If 'examples' is defined as in the training cell
    for i, ex in enumerate(examples):
        kb_texts.append(ex["response"])
        kb_ids.append(f"policy_{i}")
except NameError:
    print("WARNING: 'examples' not found; please run the dataset cell first.")
    kb_texts = ["Our customer support team is available Monday to Friday 09:00–17:00 CET."]
    kb_ids = ["fallback_policy"]

print(f"KB documents: {len(kb_texts)}")

# 2) Fit TF-IDF retriever
vectorizer = TfidfVectorizer()
kb_matrix = vectorizer.fit_transform(kb_texts)

def retrieve_docs(query: str, k: int = 3):
    """Return top-k KB snippets most similar to the query."""
    q_vec = vectorizer.transform([query])
    sims = cosine_similarity(q_vec, kb_matrix)[0]
    top_idx = np.argsort(sims)[::-1][:k]
    results = []
    for idx in top_idx:
        results.append({
            "id": kb_ids[idx],
            "text": kb_texts[idx],
            "score": float(sims[idx]),
        })
    return results

# 3) Generation pipeline (reuses the fine-tuned model if you've run training)
rag_pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer
)

def answer_with_rag(question: str, k: int = 3, max_new_tokens: int = 80) -> dict:
    """Retrieve KB snippets and generate an answer grounded in that context."""
    retrieved = retrieve_docs(question, k=k)
    context = "\n".join(f"- {r['text']}" for r in retrieved)

    prompt = (
        "System: You are a customer-support assistant. Answer the customer using ONLY the information "
        "from the context. If the answer is not in the context, say you don't know and suggest contacting support.\n"
        f"Context:\n{context}\n\n"
        f"Customer: {question}\n"
        "Agent:"
    )

    out = rag_pipe(
        prompt,
        max_new_tokens=max_new_tokens,
        do_sample=True,
        top_p=0.9,
        temperature=0.7,
        repetition_penalty=1.2,
        no_repeat_ngram_size=4,
        eos_token_id=tokenizer.eos_token_id,
        pad_token_id=tokenizer.eos_token_id,
    )[0]["generated_text"]

    if "Agent:" in out:
        answer = out.split("Agent:", 1)[-1].strip()
    else:
        answer = out.strip()

    return {
        "question": question,
        "context_snippets": retrieved,
        "answer": answer,
    }

# 4) Demo: ask a few questions and see retrieved context + grounded answer
demo_questions = [
    "How can I reset my password?",
    "What are your support opening hours?",
    "How do I return a damaged item?",
    "Do you offer volume discounts for teams?",
]

for q in demo_questions:
    result = answer_with_rag(q, k=3)
    print("\n=== QUESTION ===")
    print(q)
    print("\n--- Retrieved KB snippets ---")
    for s in result["context_snippets"]:
        print(f"[{s['id']}] (score={s['score']:.3f}) {s['text']}")
    print("\n--- Model answer (RAG) ---")
    print(result["answer"])
    print("=" * 80)



Device set to use cuda:0


KB documents: 22

=== QUESTION ===
How can I reset my password?

--- Retrieved KB snippets ---
[policy_0] (score=0.331) You can reset your password by clicking "Forgot password" on the login page. We will send a reset link to your registered email address. If you don't see it within a few minutes, please check your spam folder or contact our support team again.
[policy_2] (score=0.237) For security reasons, your account is temporarily locked after several failed login attempts. Please wait 15 minutes and try again, or use the "Forgot password" link on the login page to reset your password immediately.
[policy_16] (score=0.137) You can download a copy of your personal data under "Account settings" -> "Privacy" -> "Download my data". The export includes information such as your profile details, orders and support interactions.

--- Model answer (RAG) ---
Name: Customer: Password: My password is "2.0.1", please enter it at least once (only after 6 months).
- Contact me directly via phone:

## Deployment & CI/CD notes

- After training, save LoRA adapter weights and optionally merge with base weights or use adapter-loading at inference.
- Store artifacts in S3; deploy via SageMaker endpoint or containerized REST API behind autoscaling.
- Automate retraining triggers (e.g., new labeled data > threshold) with GitHub Actions or AWS Step Functions; include human-in-the-loop for validation of policy-sensitive outputs.
